# P2.1 — Semantic Expert FINAL (Kaggle)

This notebook is preconfigured for the frozen P2.1 final protocol. It uses source-only, label-free reconstruction data and does not claim anomaly-detection performance.

Before running:
1. Attach the Kaggle dataset containing the extracted `phase2-final` and `seqlogad-code-final` folders. Do not enter their paths manually.
2. In **Session options**, enable an NVIDIA GPU and **Internet**. Native-BF16 hardware is preferred; T4 FP16 is conditional on the finite preflight.
3. Add a Kaggle User Secret named `HF_TOKEN` and grant this notebook access. Never paste the token into a cell.
4. Run all cells from the top. Inputs stay read-only under `/kaggle/input`; code, venv, checkpoints and exports are written only under `/kaggle/working`.
5. Change only `FOLD_ID`, `SEED`, or `RESUME_FROM` for another frozen final run.

The notebook verifies the extracted code provenance and data bundle hash, copies code to a writable working directory, and supports both the original final code dataset and the RUN_ID-fixed revision.


In [ ]:
# CONFIG — frozen final protocol; change only FOLD_ID, SEED, or RESUME_FROM.
from pathlib import Path
import hashlib, json, os, re, shutil, subprocess, sys

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
OUTPUT_ROOT = str(WORK_ROOT / "seqlogad_outputs")
DATA_ROOT = None       # discovered from the trusted bundle hash
REPO_ROOT = None       # copied from verified Kaggle input to /kaggle/working
FOLD_ID = "FOLD-TARGET-ARCH-BGL"
RUN_MODE = "final"
SEED = 42              # allowed: 42, 3407, 8675309
RESUME_FROM = None     # path to an intact checkpoint, if resuming

TARGET = FOLD_ID.removeprefix("FOLD-TARGET-ARCH-")
RUN_ID = f"P2.1-FINAL-{TARGET}-S{SEED}"
TRAINING_MODE = "QLORA_NF4_DOUBLE_QUANT"
HYPERPARAMETERS = {"seed": SEED, "training_mode": TRAINING_MODE}
EXPECTED_BUNDLE_SHA256 = "3ef2bfe36d2ad68205eea4290a632eb81db0d63acfb76c92f25a3eb96e8205de"
EXPECTED_CODE_PROVENANCE_SHA256S = {
    "0e9adac595fa64acc07be66fadaad010c2af4d57ffcc6baf467bf10ce93f6ff0",  # original final package
    "325299b865d4bfc1c7bf45df3e48544de1194c3452088bcfa09970150a8befef",  # RUN_ID-fixed package
}
assert RUN_MODE == "final"
assert FOLD_ID in {"FOLD-TARGET-ARCH-HDFS", "FOLD-TARGET-ARCH-BGL", "FOLD-TARGET-ARCH-HADOOP"}
assert SEED in {42, 3407, 8675309}

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def isolated_subprocess_env():
    env = dict(os.environ, PYTHONUNBUFFERED="1")
    env.pop("PYTHONPATH", None)  # prevent Kaggle sitecustomize leaking into the isolated Python 3.12 venv
    env.pop("PYTHONHOME", None)
    return env

def run_live(args, check=True):
    env = isolated_subprocess_env()
    with subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, env=env) as process:
        try:
            for line in process.stdout:
                token = os.environ.get("HF_TOKEN")
                if token:
                    line = line.replace(token, "[TOKEN_HIDDEN]")
                print(re.sub(r"hf_[A-Za-z0-9]+", "[TOKEN_HIDDEN]", line), end="", flush=True)
            code = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill(); process.wait()
            raise
    if code and check:
        raise RuntimeError(f"Command failed (exit {code}); read the actual error above.")
    return code


## Discover trusted inputs and construct the isolated runtime

Kaggle datasets are already extracted and are read-only. This cell locates the unique matching bundle/code tree, verifies it, copies code into `/kaggle/working`, applies the historical RUN_ID compatibility repair only when needed, and creates the frozen Python 3.12 environment.


In [ ]:
assert KAGGLE_INPUT_ROOT.is_dir(), "This notebook must run on Kaggle with the DACNTT dataset attached"
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# Locate the exact data root by the frozen bundle-manifest hash.
bundle_candidates = sorted(KAGGLE_INPUT_ROOT.rglob("manifests/bundle.json"))
bundle_matches = [p for p in bundle_candidates if sha256_file(p) == EXPECTED_BUNDLE_SHA256]
assert len(bundle_matches) == 1, (
    f"Expected exactly one phase2 bundle with SHA-256 {EXPECTED_BUNDLE_SHA256}; "
    f"found {len(bundle_matches)} among {[str(p) for p in bundle_candidates]}"
)
DATA_ROOT = str(bundle_matches[0].parent.parent)

# Locate and independently verify the extracted code package.
config_candidates = sorted(KAGGLE_INPUT_ROOT.rglob("configs/models/base-freeze-v1.yaml"))
verified_code_roots = []
for config_path in config_candidates:
    root = config_path.parents[2]
    provenance_path = root / "CODE_PROVENANCE.json"
    train_path = root / "src/seqlogad/semantic/train.py"
    if not provenance_path.is_file() or not train_path.is_file():
        continue
    if sha256_file(provenance_path) not in EXPECTED_CODE_PROVENANCE_SHA256S:
        continue
    provenance = json.loads(provenance_path.read_text())
    expected_files = provenance["files"]
    actual_files = {
        p.relative_to(root).as_posix(): sha256_file(p)
        for p in root.rglob("*")
        if p.is_file() and p.name != "CODE_PROVENANCE.json"
    }
    if actual_files == expected_files:
        verified_code_roots.append(root)
assert len(verified_code_roots) == 1, (
    f"Expected exactly one verified SeqLogAD code tree; found {len(verified_code_roots)}. "
    f"Candidates: {[str(p) for p in config_candidates]}"
)
SOURCE_REPO_ROOT = verified_code_roots[0]

# Copy read-only Kaggle input to a controlled writable directory.
working_repo = WORK_ROOT / "seqlogad_code"
working_marker = working_repo / ".seqlogad_kaggle_working_copy"
if working_repo.exists():
    assert working_marker.is_file(), f"Refusing to replace unrecognized directory: {working_repo}"
    shutil.rmtree(working_repo)
shutil.copytree(SOURCE_REPO_ROOT, working_repo)
working_marker.write_text("generated from verified Kaggle input\n")
REPO_ROOT = str(working_repo)
BASE_FREEZE_CONFIG = str(working_repo / "configs/models/base-freeze-v1.yaml")

# Compatibility repair for the original final code dataset. Newer code is already fixed.
train_py = working_repo / "src/seqlogad/semantic/train.py"
source = train_py.read_text()
old_guard = """    if not run_id or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-_' for c in run_id):
        raise ValueError('unsafe run_id')
"""
fixed_guard = """    import re
    if not isinstance(run_id, str) or re.fullmatch(
            r'[A-Za-z0-9]+(?:[._-][A-Za-z0-9]+)*', run_id) is None:
        raise ValueError('unsafe run_id')
"""
if old_guard in source:
    train_py.write_text(source.replace(old_guard, fixed_guard, 1))
    run_id_guard_status = "PATCHED_IN_WORKING_COPY"
elif "RUN_ID_PATTERN = re.compile" in source and "validate_run_id(run_id)" in source:
    run_id_guard_status = "ALREADY_FIXED"
else:
    raise RuntimeError("Unrecognized RUN_ID validator; refusing to modify code")
compile(train_py.read_text(), str(train_py), "exec")

# The frozen project runtime is Python 3.12. Bootstrap it with uv only if Kaggle lacks python3.12.
python312 = sys.executable if sys.version_info[:2] == (3, 12) else shutil.which("python3.12")
if not python312:
    run_live([sys.executable, "-m", "pip", "install", "uv"], check=True)
    uv = shutil.which("uv")
    assert uv, "uv installation succeeded but its executable is not on PATH"
    run_live([uv, "python", "install", "3.12"], check=True)
    python312 = subprocess.check_output(
        [uv, "python", "find", "3.12"], text=True, env=isolated_subprocess_env()
    ).strip()
python_version = subprocess.check_output(
    [python312, "-c", "import platform; print(platform.python_version())"],
    text=True, env=isolated_subprocess_env()
).strip()
assert python_version.startswith("3.12."), f"Expected Python 3.12, got {python_version}"

VENV = WORK_ROOT / "seqlogad-venv"
PYTHON = str(VENV / "bin/python")
run_live([sys.executable, "-m", "pip", "install", "virtualenv==20.31.2"], check=True)
run_live([sys.executable, "-m", "virtualenv", "--python", python312, str(VENV)], check=True)
run_live([PYTHON, "-m", "pip", "--version"], check=True)
run_live([PYTHON, "-m", "pip", "install", "torch==2.9.1", "--index-url",
          "https://download.pytorch.org/whl/cu128"], check=True)
run_live([PYTHON, "-m", "pip", "install", "-e", REPO_ROOT, "packaging>=24,<27"], check=True)
run_live([PYTHON, "-c", "from seqlogad.semantic.runtime import install_missing; "
          + "install_missing(" + repr(REPO_ROOT) + ")"], check=True)

resolved = subprocess.check_output([
    PYTHON, "-c", "import json; from seqlogad.semantic.config import load_config; "
    + "print(json.dumps(load_config(" + repr(REPO_ROOT) + ", run_mode='final')[0]))"
], env=isolated_subprocess_env())
FROZEN = json.loads(resolved)
MODEL_ID = FROZEN["model_id"]
MODEL_REVISION = FROZEN["model_revision"]
TOKENIZER_REVISION = FROZEN["tokenizer_revision"]
print({
    "DATA_ROOT": DATA_ROOT,
    "SOURCE_REPO_ROOT": str(SOURCE_REPO_ROOT),
    "REPO_ROOT": REPO_ROOT,
    "OUTPUT_ROOT": OUTPUT_ROOT,
    "python": python_version,
    "run_id_guard": run_id_guard_status,
    "MODEL_ID": MODEL_ID,
    "MODEL_REVISION": MODEL_REVISION,
    "TOKENIZER_REVISION": TOKENIZER_REVISION,
})


## GPU/CUDA/VRAM and frozen runtime check

This check must pass before authentication or model loading. Kaggle may assign different GPU types; FP16 hardware is accepted only if the later forward/backward preflight is finite.


In [ ]:
def command(action, *extra):
    return [PYTHON, "-m", "seqlogad.semantic.cli", action, "--repo-root", REPO_ROOT,
            "--data-root", DATA_ROOT, "--output-root", OUTPUT_ROOT, "--fold-id", FOLD_ID,
            "--run-id", RUN_ID, "--run-mode", RUN_MODE, *extra]

run_live(command("environment"), check=True)


## Kaggle Secret authentication and frozen revision verification

Create the `HF_TOKEN` secret in Kaggle and enable notebook access. Its value is never printed or saved.


In [ ]:
if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        raise RuntimeError(
            "Add a Kaggle User Secret named HF_TOKEN and grant this notebook access"
        ) from None
assert os.environ.get("HF_TOKEN"), "HF_TOKEN Kaggle Secret is empty"
run_live(command("auth"), check=True)


## Manifest, checksum and leakage validation

The attached data remains under read-only `/kaggle/input`. Validation checks the trusted bundle digest, complete allowlist, hashes, source-only roles, Phase-1 membership and forbidden fields. No anomaly labels are used.


In [ ]:
assert Path(DATA_ROOT, "manifests/bundle.json").is_file()
run_live(command("validate", "--expected-bundle-sha256", EXPECTED_BUNDLE_SHA256), check=True)
OVERRIDES = str(WORK_ROOT / "semantic-run-config.json")
Path(OVERRIDES).write_text(json.dumps(HYPERPARAMETERS))
print({
    "DATA_ROOT": DATA_ROOT,
    "OUTPUT_ROOT": OUTPUT_ROOT,
    "FOLD_ID": FOLD_ID,
    "RUN_ID": RUN_ID,
    "hyperparameters": HYPERPARAMETERS,
    "resume": RESUME_FROM,
})


## Final QLoRA training

The Python modules resolve the frozen final budget: 8192 train and 512 validation records per source, maximum 1536 steps, seed-specific deterministic selection, validation/checkpoint every 100 steps, and early stopping on aggregate validation reconstruction NLL with patience 3 and minimum delta 0.001. No anomaly labels are used.

The first real model step runs a finite forward/backward preflight. Any OOM or non-finite loss aborts rather than silently changing the scientific configuration.


In [ ]:
assert Path(DATA_ROOT, "manifests/bundle.json").is_file(), "Run data validation first"
run_directory = Path(OUTPUT_ROOT) / FOLD_ID / "SEMANTIC_LLAMA" / RUN_ID
assert not run_directory.exists(), f"Run directory already exists: {run_directory}"
args = command("train", "--config-json", OVERRIDES,
               "--expected-bundle-sha256", EXPECTED_BUNDLE_SHA256)
if RESUME_FROM:
    args += ["--resume-from", RESUME_FROM]
run_live(args, check=True)


## Selected checkpoint, metrics and sanity evidence

These are reconstruction and engineering sanity outputs, not anomaly-detection performance.


In [ ]:
RUN_DIR = Path(OUTPUT_ROOT) / FOLD_ID / "SEMANTIC_LLAMA" / RUN_ID
for name in ["metrics.json", "coverage.json", "selection.json", "sanity-evidence.json",
             "semantic-probes.json", "manifest.json", "checksums.sha256"]:
    path = RUN_DIR / name
    print(f"\n--- {name} ---")
    if name.endswith(".json"):
        print(json.dumps(json.loads(path.read_text()), indent=2))
    else:
        print(path.read_text())


## Export the complete run

The ZIP remains in `/kaggle/working`, appears under notebook Output after **Save Version**, and can also be downloaded from the link displayed below. Preserve the entire run, including all checkpoint siblings and ledgers.


In [ ]:
from IPython.display import FileLink, display

archive = shutil.make_archive(str(WORK_ROOT / "seqlogad_outputs"), "zip", OUTPUT_ROOT)
print({"export": archive, "sha256": sha256_file(archive)})
display(FileLink(archive))
